# HyperStreamDB: 15-Minute Live Demo (Part 1)
## Installation & Basics

Welcome to the HyperStreamDB live demo! This notebook covers:
1.  **Installation** from source.
2.  **Loading real data** (AG News dataset).
3.  **Ingesting** data into HyperStreamDB with vector embeddings.
4.  **Scalar Filtering & Vector Search**.
5.  **Hybrid Search** (Combining both).

### 1. Installation & Setup

HyperStreamDB is built in Rust with high-performance Python bindings via `maturin`. In a production environment, you would simply `pip install hyperstreamdb`.

In [ ]:
!pip install maturin
!cd .. && maturin develop

📦 Including license file `LICENSE`
🍹 Building a mixed python/rust project
🔗 Found pyo3 bindings
🐍 Found CPython 3.12 at /home/ralbright/projects/hyperstreamdb/venv/bin/python
📡 Using build options features, bindings from pyproject.toml
Ignoring pytest: markers 'extra == "dev"' don't match your environment
Ignoring pytest-asyncio: markers 'extra == "dev"' don't match your environment
Ignoring maturin: markers 'extra == "dev"' don't match your environment
   Compiling pyo3-build-config v0.26.0
   Compiling pyo3-ffi v0.26.0=======>  ] 811/857: pyo3-build-config           
   Compiling pyo3-macros-backend v0.26.0
   Compiling pyo3 v0.26.0
   Compiling numpy v0.26.0
   Compiling pyo3-macros v0.26.0====>  ] 821/857: pyo3-macros-backend         
   Compiling arrow-pyarrow v57.3.0> ] 823/857: pyo3                           
   Compiling pythonize v0.26.0
   Compiling arrow v57.3.0========> ] 823/857: arrow-pyarrow, pyo3, python…
   Compiling datafusion-common v52.4.0 ] 824/857: arrow, arrow-py

In [12]:
!source ~/.bashrc
!pip install ipywidgets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.8/139.8 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 914.9/914.9 kB 8.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 14.4 MB/s eta 0:00:00a 0:00:01


In [ ]:
import warnings
warnings.filterwarnings('ignore', category=UserWarning)
import os
from huggingface_hub import login
HF_TOKEN = os.environ.get("HF_TOKEN")
login(token=HF_TOKEN)

In [5]:
import hyperstreamdb as hdb
import pandas as pd
import numpy as np
from datasets import load_dataset
from sentence_transformers import SentenceTransformer

In [6]:
# Initialize the Intel GPU context
ctx = hdb.ComputeContext("intel")

### 2. Load Real Data

We use the **AG News** dataset, which contains 4 categories of news articles (World, Sports, Business, Sci/Tech).

In [ ]:
# Load 500 test articles
dataset = load_dataset("ag_news", split="test[:500]")
df = pd.DataFrame(dataset)

# Map numeric labels to names
label_map = {0: "World", 1: "Sports", 2: "Business", 3: "Sci/Tech"}
df["category"] = df["label"].map(label_map)

df.head()

,text,label,category
0,Fears for T N pension after talks Unions repre...,2,Business
1,The Race is On: Second Private Team Sets Launc...,3,Sci/Tech
2,Ky. Company Wins Grant to Study Peptides (AP) ...,3,Sci/Tech
3,Prediction Unit Helps Forecast Wildfires (AP) ...,3,Sci/Tech
4,Calif. Aims to Limit Farm-Related Smog (AP) AP...,3,Sci/Tech


### 3. Ingest into HyperStreamDB

We'll embed the article text and store it in a HyperStreamDB table. HyperStreamDB uses **Overlay Indexing**, storing HNSW and Inverted indexes as sidecar files alongside standard Parquet.

In [ ]:
# Initialize embedding model (local, small, fast)
model = SentenceTransformer("all-MiniLM-L6-v2")

print("Embedding 500 articles...")
embeddings = model.encode(df["text"].tolist())
df["embedding"] = [list(e) for e in embeddings]

# Create HyperStreamDB table
import shutil, os
if os.path.exists("news_db"):
    shutil.rmtree("news_db")
table = hdb.Table("news_db", context=ctx)

# Enable Vector Indexing for the embedding column
table.add_index_columns(["embedding", "category"])

print("Ingesting into HyperStreamDB...")
table.write_pandas(df)
table.commit()
print("Done!")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4459.63it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding 500 articles...
Ingesting into HyperStreamDB...
Done!


### 4. Scalar Filtering

HyperStreamDB's **Inverted Index** allows for extremely fast scalar filtering (using RoaringBitmaps).

In [5]:
# Filter to only 'Sports' articles
sports_articles = table.to_pandas(filter="category = 'Sports'")
print(f"Found {len(sports_articles)} sports articles.")
sports_articles[["category", "text"]].head()

Found 145 sports articles.


,category,text
0,Sports,Giddy Phelps Touches Gold for First Time Micha...
1,Sports,Tougher rules won't soften Law's game FOXBOROU...
2,Sports,Shoppach doesn't appear ready to hit the next ...
3,Sports,Mighty Ortiz makes sure Sox can rest easy Just...
4,Sports,They've caught his eye In quot;helping themse...


### 5. Vector Search

We search for semi-similar articles using our query embedding.

In [7]:
query = "Winning medals in international sports competitions"
query_embedding = list(model.encode(query))

# Vector search (top-5)
results = table.filter(vector_filter=query_embedding, k=5).to_pandas()
results[["category", "text", "_distance"]]

AttributeError: 'builtins.Table' object has no attribute 'filter'

### 6. Hybrid Search

Combine **Scalar Filter** with **Vector Search** seamlessly.

In [ ]:
# "Sci/Tech articles most similar to this query"
hybrid_results = table.to_pandas(
    filter="category = 'Sci/Tech'",
    vector_filter=query_embedding,
    k=5
)
hybrid_results[["category", "text", "_distance"]]